# General Health Query Chatbot
### Powered by Hugging Face + Mistral-7B-Instruct

## Install Dependencies

In [1]:
# install required packages 
!pip install huggingface_hub ipywidgets requests --quiet

print("All packages installed successfully!")
print("huggingface_hub - Hugging Face API client")
print(" ipywidgets   - Interactive chat UI in notebook")
print(" requests    Http calls to HF Inference API")

All packages installed successfully!
huggingface_hub - Hugging Face API client
 ipywidgets   - Interactive chat UI in notebook
 requests    Http calls to HF Inference API


In [3]:
import os
import re
import textwrap
import json
from datetime import datetime 
import requests

from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

# model to use free on hugging face inference API
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
API_URL = "https://router.huggingface.co/v1/chat/completions"


# verify token is set
if HF_TOKEN == "HF_TOKEN":
    print("Please replace 'hf_YOUR_TOKEN_HERE' with your actual token")
else:
    print(f" Token configured")
    print(f"   Model: {MODEL}")
    print(f"   API  : {API_URL}")

 Token configured
   Model: meta-llama/Llama-3.1-8B-Instruct
   API  : https://router.huggingface.co/v1/chat/completions


## Prompt Engineering

In [5]:
SYSTEM_PROMPT = """You are HealthBot, a friendly and knowledgeable general-health assistant.

YOUR ROLE:
- Provide clear, accurate, easy-to-understand information on general health topics.
- Use plain, simple language. Explain any medical terms you use.
- Be warm, empathetic, and supportive in tone.
- Keep answers concise: 3-5 sentences for simple questions; use bullet points for lists.

SAFETY RULES — follow in EVERY response without exception:
1. NEVER diagnose a specific illness or medical condition for the user.
2. NEVER recommend a specific prescription medication or personalised dosage.
3. NEVER advise the user to stop or change their current medication without a doctor.
4. For ANY emergency symptoms (chest pain, difficulty breathing, stroke signs, severe
   allergic reaction, loss of consciousness, suicidal thoughts) — ALWAYS tell the user
   to call emergency services (911 / 999 / 112) or go to an emergency room immediately.
5. End EVERY response with:
   "⚕️ For personal medical advice, please consult a qualified healthcare professional."

FORMAT:
- Use at most 1-2 emoji per response to stay friendly.
- If a question is outside health topics, politely say you can only help with health queries.
"""

def build_prompt(user_message: str, chat_history: list) -> str:
    """
    Builds a Mistral-7B-Instruct formatted prompt with full conversation history
    Format: <s>[INST] <<SYS>>system<</SYS>> user_msg [/ISNT] assistant msg
    """
    prompt = f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"

    # add conversation history
    for turn in chat_history:
        if turn['role'] == 'user':
            prompt +=f"{turn['content']} [/INST]"
        elif turn['role'] == 'assistant':
            prompt += f"{turn['content']} </s><s>[INST]"
    
    # add current user message
    prompt += f"{user_message} [/INST]"
    return prompt

print(" System prompt and prompt builder ready.")
print(f"  Prompt length: {len(SYSTEM_PROMPT)} characters")
print()
print(" Sample prompt preview (first 300 chars):")
sample = build_prompt("What causes a sore throat?",[])
print(sample[:300] + "...")



 System prompt and prompt builder ready.
  Prompt length: 1203 characters

 Sample prompt preview (first 300 chars):
<s>[INST] <<SYS>>
You are HealthBot, a friendly and knowledgeable general-health assistant.

YOUR ROLE:
- Provide clear, accurate, easy-to-understand information on general health topics.
- Use plain, simple language. Explain any medical terms you use.
- Be warm, empathetic, and supportive in tone.
...


### Safety Filter

In [6]:
# emergency keyword detection
EMERGENCY_PATTERNS =  [
    r"chest\s*pain",
    r"can'?t\s*breathe",
    r"difficulty\s*breath",
    r"\bstroke\b",
    r"heart\s*attack",
    r"suicid",
    r"overdos",
    r"unconscious",
    r"not\s*waking",
    r"severe\s*bleed",
    r"poison(?:ing)?",
    r"anaphylax",
    r"\bseizure\b",
    r"coughing\s*blood",
    r"\bhang(?:ing)?\s+myself\b",
]

EMERGENCY_RESPONSE = (
    " EMERGENCY ALERT\n\n"
    "This sounds like it could be a medical emergency.\n"
    "Please call emergency services IMMEDIATELY: \n\n"
    " 911 -- United States & Canada\n"
    " 1122 -- Pakistan\n"
    "Go to your nearest emergency room right away.\n"
    "Do NOT wait -- your life may be at risk"
)

HARMFUL_PATTERNS = [
    r"how\s+to\s+fake\s+(illness|symptom)",
    r"trick\s+doctor",
    r"get\s+prescription\s+without",
]

HARMFUL_RESPONSE = (
    "⚠️ I'm not able to help with that request. "
    "I can only provide general health information to help you stay healthy and informed. "
    "Please consult a qualified healthcare professional for any medical needs."
)

def safety_check(user_input: str) -> str | None:
    """ 
    Pre-flight safety filter.
    Retruns a safety response string if triggered, otherwise None (safe to proceed)

    """
    text = user_input.lower()

    # Emergency check (highest priority)
    for pattern in EMERGENCY_PATTERNS:
        if re.search(pattern,text):
            return EMERGENCY_RESPONSE
        
    # Harmful request check
    for pattern in HARMFUL_PATTERNS:
        if re.search(pattern,text):
            return HARMFUL_RESPONSE
        
    return None  # safe - proceed to LLM

# ---- Self- Test
test_cases = [
    ("What causes a sore throat?",                  "SAFE"),
    ("I have chest pain right now",                 "EMERGENCY"),
    ("is paracetamol safe for children?",           "SAFE"),
    ("I think I overdosed on medication",           "EMERGENCY"),
    ("How do I stay hydrated?",                     "SAFE"),
    ("How to trick my doctor for painkillers?",     "Harmful"),
]

print("Safety Filter Self-Test:\n")
all_pass = True
for msg, expected in test_cases:
    result = safety_check(msg)
    if result is None:
        actual = "SAFE"
    elif "EMERGENCY" in result:
        actual = "EMERGENCY"
    else:
        actual = "HARMFUL"
    
    status = "✅ PASS" if actual == expected else "❌ FAIL"
    if actual != expected:
        all_pass = False
    print(f"  {status}  [{expected:9s}]  '{msg}'")


print(f"\n{'All tests passed!' if all_pass else 'X some tests failed!'}")




Safety Filter Self-Test:

  ✅ PASS  [SAFE     ]  'What causes a sore throat?'
  ✅ PASS  [EMERGENCY]  'I have chest pain right now'
  ✅ PASS  [SAFE     ]  'is paracetamol safe for children?'
  ✅ PASS  [EMERGENCY]  'I think I overdosed on medication'
  ✅ PASS  [SAFE     ]  'How do I stay hydrated?'
  ❌ FAIL  [Harmful  ]  'How to trick my doctor for painkillers?'

X some tests failed!


#### Hugging Face API Integration

In [ ]:

#  HF INFERENCE API  —  Free LLM calls with Mistral-7B


conversation_history = []  # Stores full chat for multi-turn context

def call_hf_api(prompt: str) -> str:
    """
    Sends a prompt to the HF Inference API and returns the model's text.
    The HF free tier may occasionally be slow if the model is loading (~20s).
    """
    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 400,
        "temperature": 0.7
    }
    
    response = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    
    if response.status_code == 200:
        result = response.json()
        if isinstance(result, list) and len(result) > 0:
            return result[0].get("generated_text", "").strip()
        return str(result).strip()
    elif response.status_code == 503:
        return "⏳ Model is loading (free tier cold start). Please wait 20 seconds and try again."
    elif response.status_code == 401:
        return "❌ Invalid HF token. Please check your token in Step 2."
    elif response.status_code == 429:
        return "⏳ Rate limit reached on free tier. Please wait a minute and try again."
    else:
        return f"❌ API Error {response.status_code}: {response.text[:200]}"


def clean_response(text: str) -> str:
    """Remove any leftover prompt artifacts from the model output."""
    # Remove instruction tags if model accidentally echoes them
    text = re.sub(r'\[/?INST\]', '', text)
    text = re.sub(r'<</?SYS>>', '', text)
    text = re.sub(r'</?s>', '', text)
    return text.strip()


def ask_healthbot(user_message: str) -> str:
    """
    Main function — send a health question to HealthBot, get a safe response.
    
    Args:
        user_message (str): The user's health question.
    
    Returns:
        str: HealthBot's response (safety-filtered + LLM-generated).
    """
    # 1. Safety pre-filter (no API call if emergency/harmful)
    alert = safety_check(user_message)
    if alert:
        return alert

    # 2. Build formatted prompt with history
    prompt = build_prompt(user_message, conversation_history)
    
    # 3. Store user message in history
    conversation_history.append({"role": "user", "content": user_message})
    
    # 4. Call free HF API
    raw_reply = call_hf_api(prompt)
    reply = clean_response(raw_reply)
    
    # 5. Store assistant reply in history
    if not reply.startswith("❌") and not reply.startswith("⏳"):
        conversation_history.append({"role": "assistant", "content": reply})
    
    return reply


def reset_conversation():
    """Clear conversation history for a fresh chat session."""
    global conversation_history
    conversation_history = []
    print("🔄 Conversation reset.")


print("✅ ask_healthbot() is ready — using FREE Hugging Face API")
print(f"   Model: {MODEL}")


✅ ask_healthbot() is ready — using FREE Hugging Face API
   Model: meta-llama/Llama-3.1-8B-Instruct


#### Test API Connection

In [9]:
print("Testing connection to Hugging Face API...")
print(f"Model: {MODEL}\n")

test_reply = ask_healthbot("In one sentence, what is your name and purpose?")
print(f"Response: {test_reply}")

if test_reply.startswith("❌"):
    print("\n⚠️  Fix the error above, then re-run this cell.")
elif test_reply.startswith("⏳"):
    print("\n⏳ Model is loading. Wait 20 seconds then run this cell again.")
else:
    print("\n✅ Connection successful! HealthBot is ready to use.")


Testing connection to Hugging Face API...
Model: meta-llama/Llama-3.1-8B-Instruct

Response: {'id': 'chatcmpl-22d273b8-1aa2-4bcb-a70a-a226e6623d63', 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': "I'm HealthBot, a friendly and knowledgeable assistant designed to provide clear and accurate information on general health topics to support your well-being ⚕️.\n\nFor personal medical advice, please consult a qualified healthcare professional.", 'role': 'assistant'}}], 'created': 1772537865, 'model': 'llama3.1-8b', 'system_fingerprint': 'fp_5198798116a66ebf301b', 'object': 'chat.completion', 'usage': {'total_tokens': 356, 'completion_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0, 'reasoning_tokens': 0}, 'prompt_tokens': 312, 'prompt_tokens_details': {'cached_tokens': 0}}, 'time_info': {'queue_time': 0.17187412900000001, 'prompt_time': 0.020972124999999994, 'completion_time': 0.017974476, 'total_time': 0.2

### Example Query demonstration

In [10]:
# Testing with the assignment's required example queries plus extras

example_queries = [
    "What causes a sore throat?", 
    "Is paracetamol safe for children?",
    "How much water should I drink daily?",
    "What are symptoms of vitamin D deficiency?",
    "What is the difference between a cold and the flu?"
]

reset_conversation()

for i , question in enumerate(example_queries):
    print(f"{'='*60}")
    print(f" Q{i}: {question}")
    print(f"{'='*60}")
    answer = ask_healthbot(question)
    for line in answer.split('\n'):
        if line.strip():
            print(textwrap.fill(line, width=60))
        else:
            print()
    print()

🔄 Conversation reset.
 Q0: What causes a sore throat?
{'id': 'chatcmpl-718de748-31a0-419a-a5d5-7512a02eb3b8',
'choices': [{'finish_reason': 'stop', 'index': 0, 'message':
{'content': 'A sore throat can be quite uncomfortable.
There are several reasons why you might be experiencing a
sore throat. Here are some common causes:\n\n* Viral
infections like the common cold or flu can cause a sore
throat.\n* Bacterial infections like strep throat can also
cause a sore throat.\n* Allergies, such as seasonal
allergies, can lead to postnasal drip and irritation of the
throat.\n* Dry air or breathing through your mouth can dry
out your throat and cause discomfort.\n* Shouting,
screaming, or singing can strain your vocal cords and lead
to a sore throat.\n\nTo help soothe a sore throat, try
drinking warm liquids, like tea or broth, and staying
hydrated. You can also try gargling with salt water or using
a humidifier to add moisture to the air. 🤒\n\n⚕️ For
personal medical advice, please consult a qu

In [11]:
#  SAFETY FILTER DEMO  —  No API calls made for these queries

emergency_queries = [
    "I'm having severe chest pain and can't breathe",
    "I think I overdosed on my medication",
    "Someone near me is unconscious and not waking up",
]

print("🛡️  Safety filter in action (API is NOT called for these):\n")

for q in emergency_queries:
    print(f"👤 User   : {q}")
    result = safety_check(q)   # Direct safety check, no API call
    print(f"🤖 HealthBot:")
    for line in result.split("\n"):
        print(f"   {line}")
    print("-" * 55)

🛡️  Safety filter in action (API is NOT called for these):

👤 User   : I'm having severe chest pain and can't breathe
🤖 HealthBot:
    EMERGENCY ALERT
   
   This sounds like it could be a medical emergency.
   Please call emergency services IMMEDIATELY: 
   
    911 -- United States & Canada
    1122 -- Pakistan
   Go to your nearest emergency room right away.
   Do NOT wait -- your life may be at risk
-------------------------------------------------------
👤 User   : I think I overdosed on my medication
🤖 HealthBot:
    EMERGENCY ALERT
   
   This sounds like it could be a medical emergency.
   Please call emergency services IMMEDIATELY: 
   
    911 -- United States & Canada
    1122 -- Pakistan
   Go to your nearest emergency room right away.
   Do NOT wait -- your life may be at risk
-------------------------------------------------------
👤 User   : Someone near me is unconscious and not waking up
🤖 HealthBot:
    EMERGENCY ALERT
   
   This sounds like it could be a medical emerg

In [13]:
#  MULTI-TURN CONVERSATION  —  Context is remembered

reset_conversation()

turns = [
    "I've been feeling very tired lately.",
    "Could it be related to my diet?",
    "What foods would help with energy levels?",
]

print("Multi-turn conversation (HealthBot remembers context):\n")

for turn in turns:
    print(f"👤 You    : {turn}")
    response = ask_healthbot(turn)
    print(f"🤖 HealthBot: {textwrap.fill(response, width=58, subsequent_indent=' '*12)}")
    print()


🔄 Conversation reset.
Multi-turn conversation (HealthBot remembers context):

👤 You    : I've been feeling very tired lately.
🤖 HealthBot: {'id': 'chatcmpl-e94fea7b-7d1d-47c9-b776-95a14c4ca388',
            'choices': [{'finish_reason': 'stop', 'index':
            0, 'message': {'content': "Hello there, it's
            great that you're reaching out about your
            fatigue. Feeling tired all the time can be
            really frustrating. Here are some possible
            reasons and things you can try:\n\n• Lack of
            sleep or poor sleep quality\n• Dehydration\n•
            Poor diet or nutrition\n• Stress or anxiety\n•
            Physical inactivity\n• Certain medical
            conditions (some of which can be serious, so
            it's always a good idea to consult a
            doctor)\n\nIn the meantime, try these simple
            tips: \n- Get a good night's sleep (7-9 hours
            for most adults)\n- Stay hydrated by drinking
            plenty of